In [ ]:
import json

def get_secret(secret_name, default_value=None):
    try:
        return dbutils.secrets.get(scope="mtg-pipeline", key=secret_name)
    except Exception:
        if default_value is not None:
            return default_value
        raise

S3_BUCKET = get_secret("s3_bucket")
S3_STAGE_PREFIX = get_secret("s3_stage_prefix", "stage")
S3_BASE_PATH = f"s3://{S3_BUCKET}/{S3_STAGE_PREFIX}"
rulings_path = f"{S3_BASE_PATH}/rulings"

result = {"base": rulings_path, "dirs": []}
for entry in dbutils.fs.ls(rulings_path):
    dir_info = {"path": entry.path, "size": entry.size, "parts": [], "error": None}
    if entry.path.rstrip("/").endswith(".parquet"):
        try:
            inner = dbutils.fs.ls(entry.path)
            dir_info["parts"] = [{"path": f.path, "size": f.size} for f in inner]
        except Exception as e:
            dir_info["error"] = str(e)
    result["dirs"].append(dir_info)

dbutils.notebook.exit(json.dumps(result))